In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.optimizers import Adam
from tensorflow.keras import layers
from tensorflow.keras.metrics import AUC, Recall, Precision
from tensorflow.keras.models import Model
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight

In [2]:
df = pd.read_csv("Dataset_NN.csv")
y = df['presence'].values
X = df.drop(columns='presence').values

In [3]:
Y = []
for i in range(len(y)):
    if(i % 125 == 0):
        Y.append(y[i])
Y = np.array(Y)

In [4]:
scaler = MinMaxScaler(feature_range=(0, 1))
X = scaler.fit_transform(X).reshape(X.shape[0]//125, 125, X.shape[1])
X_train, X_test, y_train, y_test = train_test_split(X, Y, test_size=0.2, shuffle=False)

In [5]:
X_train[0].shape

(125, 70)

In [11]:
np.savetxt("sample.txt", X_train[10].reshape(-1), fmt="%.8f")

In [10]:
X_train[0]

array([[0.2       , 0.4       , 0.4       , ..., 0.        , 0.        ,
        0.84745763],
       [0.6       , 0.8       , 0.        , ..., 0.        , 0.        ,
        0.84745763],
       [1.        , 0.4       , 0.6       , ..., 0.        , 0.        ,
        0.84745763],
       ...,
       [0.        , 0.        , 1.        , ..., 0.        , 0.        ,
        0.86440678],
       [0.        , 0.8       , 0.2       , ..., 0.        , 0.        ,
        0.86440678],
       [1.        , 0.8       , 0.        , ..., 0.        , 0.        ,
        0.86440678]])

In [189]:
class_weights = compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
class_weight_dict = dict(zip(np.unique(y_train), class_weights))
sample_weights_train = np.array([class_weight_dict[label] for label in y_train])

In [190]:
def build_model(seq_len, num_features):
    inputs = layers.Input(shape=(seq_len, num_features))
    
    # TCN часть: Разложенные слои с растущим dilation
    x = layers.Conv1D(64, kernel_size=3, padding='causal', dilation_rate=1, activation='relu')(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.2)(x)
    
    x = layers.Conv1D(64, kernel_size=3, padding='causal', dilation_rate=2, activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.2)(x)
    
    x = layers.Conv1D(64, kernel_size=3, padding='causal', dilation_rate=4, activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.2)(x)
    
    x = layers.Conv1D(64, kernel_size=3, padding='causal', dilation_rate=8, activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.2)(x)


    mha = layers.MultiHeadAttention(num_heads=4, key_dim=64)
    attention_output, attention_scores = mha(query=x, value=x, key=x, return_attention_scores=True)
    
    pooled = layers.GlobalAveragePooling1D()(attention_output)
    
    classification = layers.Dense(1, activation='sigmoid')(pooled)  # Бинарный выход
    
    model = Model(inputs=inputs, outputs={'classification': classification, 'attention': attention_scores})
    model.compile(
        optimizer=Adam(learning_rate=0.001),
        loss={'classification': 'binary_crossentropy', 'attention': None},
        metrics={
            'classification': [
                'accuracy',  # Оставляем для совместимости
                AUC(name='auc'),  # AUC (ROC-AUC по умолчанию)
                Precision(name='precision'),  # Precision
                Recall(name='recall'),  # Recall
            ]
        }
    )
    return model

In [191]:
model = build_model(X_train.shape[1], X_train.shape[-1])
model.summary()
history = model.fit(
    X_train, 
    {'classification': y_train}, 
    epochs=50, 
    batch_size=8, 
    validation_data=(X_test, {'classification': y_test}),
    sample_weight={'classification': sample_weights_train}
)

Model: "model_21"
__________________________________________________________________________________________________
 Layer (type)                Output Shape                 Param #   Connected to                  
 input_27 (InputLayer)       [(None, 125, 70)]            0         []                            
                                                                                                  
 conv1d_124 (Conv1D)         (None, 125, 64)              13504     ['input_27[0][0]']            
                                                                                                  
 batch_normalization_72 (Ba  (None, 125, 64)              256       ['conv1d_124[0][0]']          
 tchNormalization)                                                                                
                                                                                                  
 dropout_89 (Dropout)        (None, 125, 64)              0         ['batch_normalization_7